## Self notes — running log

Short observations only. One markdown cell per note. Add a code cell underneath if a demo is genuinely useful — most of the time it isn't, the practice notebooks already cover that.


### 1 — `SparkSession` has two versions

- `pyspark.sql.session.SparkSession` — **classic**, drives a local JVM.
- `pyspark.sql.connect.session.SparkSession` — **Connect**, talks to a remote Spark server over gRPC.

Same API surface, different classes. Helpers that need to touch the JVM directly (e.g. `delta.configure_spark_with_delta_pip(builder)`) accept only the classic builder and reject the Connect one with a type-mismatch error.


### 2 — Predicate pushdown: what does and doesn't push

- **In-memory sources can never push filters.** `Range` (`spark.range`), `LocalTableScan` / `ExistingRDD` (`createDataFrame`) have no scan layer to push into — the data is already in memory. The `Filter` just sits above the source.
- **Even when a filter pushes, the `Filter` node stays in the plan.** Source-level pushdown is best-effort (e.g. Parquet uses row-group min/max stats — it can skip whole groups but rows inside surviving groups still need exact re-checking). Catalyst keeps the `Filter` above the scan for correctness.

```text
*(1) Project [customer_id#0, full_name#1, credit_score#3L]
  +- *(1) Filter (isnotnull(credit_score#3L) AND (credit_score#3L >= 700))   ← exact re-check
     +- *(1) ColumnarToRow
        +- FileScan parquet [...]
             PushedFilters: [IsNotNull(credit_score), GreaterThanOrEqual(credit_score,700)]
             ReadSchema: struct<customer_id:string,full_name:string,credit_score:bigint>
```


### 3 — `foreach(print)` vs `show()`

Both are actions, but they print from different places and behave very differently:

| | `df.show()` | `df.foreach(print)` |
|---|---|---|
| Where the print happens | Driver | Executors |
| Where you see the output | Your console | Executor stdout / cluster logs — usually invisible in a notebook or `spark-submit --master yarn` |
| Output format | Pretty ASCII table, header + 20 rows by default | One `Row(...)` per line, unordered, no header |
| Order | First 20 rows in source order (top-down) | Whatever order partitions emit — interleaved across executors |
| Data movement | First N rows pulled to driver | None — runs on each row in place |
| Safe at scale | Yes (bounded) | Yes (no driver memory pressure), but log volume can be huge |

In `local[*]` mode the executor *is* the driver, so `foreach(print)` happens to land in the same console — which is why it looked equivalent to `show()` in the practice script. On a real cluster the prints disappear into executor logs and you'd think nothing happened.

Rule of thumb: use `show()` for inspecting data; reserve `foreach` for genuine per-row side effects (writing to a sink, pushing to a queue) and even then prefer `foreachPartition` to amortize connection setup.


---

_New entries go below. Template:_

```
### N — title

One- or two-sentence observation. Code only if it adds something the practice notebooks don't already show.
```
